# Compare datasets
This notebook compares the aggregate statistics of all tracks in every condition to ensure that the filtered `.npz` files (generated from `load_and_filter_tracks.py`) match Harvey's corrected data (filtered directly from `DIR_30S` and `DIR_5s` using the code contained below).

This confirms that the results will be consistent whether one chooses to use the `.npz` files or not (since existing scripts use both formats).

In [12]:
import os
import glob
import re
import numpy as np
import pandas as pd
from scipy.stats import chi2
from tqdm.auto import tqdm

In [19]:
DIR_30S = '/mnt/md1/jjusuf/synEP/export_qc_filtered_30s_WithCorrectedMS2_20260407'
DIR_5S = '/mnt/md1/jjusuf/synEP/export_qc_filtered_5s_WithCorrectedMS2_20260407'
NPZ_DIR = '/mnt/md1/jjusuf/synEP/data_consolidated_npz/filtered_data'
significance_level = 1e-7  # for filtering

# # on rosalind (Harvey's copy)
# DIR_30S = '/mnt/md1/Harvey/export_qc_filtered_30s_WithCorrectedMS2_20260407'
# DIR_5S = '/mnt/md1/Harvey/export_qc_filtered_5s_WithCorrectedMS2_20260407'
# NPZ_DIR = '/mnt/md1/Harvey/filtered_data'  # may be outdated


In [20]:
CONDITION_MAPPING = {
    'G7B8G2_100uM_IAA_added2hrBefore': 'G7B8G2_GSK_100uM_IAAadded2hrBefore',
    'G7B8G2_GSK_IAA(100uM)added2hrBefore': 'G7B8G2_GSK_100uM_IAAadded2hrBefore',
    'G7B8G2_GSK_IAA(100uM)added3hrBefore': 'G7B8G2_GSK_100uM_IAAadded2hrBefore',
    'G7B8G2_GSK_IAA(100uM)added4hrBefore': 'G7B8G2_GSK_100uM_IAAadded2hrBefore',
    'G7B8G2_GSK_IAA(100uM)added5hrBefore': 'G7B8G2_GSK_100uM_IAAadded2hrBefore',
    'G7B8G2_GSK_IAA(100uM)added6hrBefore': 'G7B8G2_GSK_100uM_IAAadded2hrBefore',
    'G7B8G2_GSK_IAA(100uM)added7hrBefore': 'G7B8G2_GSK_100uM_IAAadded2hrBefore',
    'G7B8G2_GSK_500uMdTAG13added2hrBefore': 'G7B8G2_GSK_500nMdTAG13added2hrBefore',
    'G7B8G2_GSK_500nmdTAG13added2hrBefore': 'G7B8G2_GSK_500nMdTAG13added2hrBefore',
    'G7B8G2_GSK_dTAG13(500uM)added3hrBefore': 'G7B8G2_GSK_500nMdTAG13added2hrBefore',
    'G7B8G2_GSK_dTAG13(500uM)added4hrBefore': 'G7B8G2_GSK_500nMdTAG13added2hrBefore',
    'G7B8G2_GSK_dTAG13(500uM)added5hrBefore': 'G7B8G2_GSK_500nMdTAG13added2hrBefore',
    'G7B8G2_GSK_dTAG13(500uM)added6hrBefore': 'G7B8G2_GSK_500nMdTAG13added2hrBefore',
    'G7B8G2_GSK_dTAG13(500uM)added7hrBefore': 'G7B8G2_GSK_500nMdTAG13added2hrBefore',
    'G7B8G2_GSK_dTAG13(500uM)added2hrBefore': 'G7B8G2_GSK_500nMdTAG13added2hrBefore',
    'S+V-A6B8': 'S+V-A6B8_GSK',
    '15A-A6G9': '15A-A6G9_GSK',
    '15A-A6_G9_GSK': '15A-A6G9_GSK',
    '15A_A6_G9_GSK': '15A-A6G9_GSK',
    '15B18G9': '15B-18G9_GSK',
    '15B-18G9': '15B-18G9_GSK',
    '15B18B9': '15B-18G9_GSK',
    '15B18G9_GSK': '15B-18G9_GSK',
    '15B-18G9_GSK': '15B-18G9_GSK',
    'S-G5H7_GSK': 'SCre-G5-H7_GSK',
    'SCreG5H7': 'SCre-G5-H7_GSK',
    'SCre-G5H7_GSK': 'SCre-G5-H7_GSK',
    'VCreC2b_GSK': 'VCre-C2b_GSK'
}

NPZ_CONDITION_MAPPING = {
    # --- 1.5kb (14B5-F10) ---
    '1.5kb_None': '14B5-F10',
    '1.5kb_IAA': '14B5-F10_GSK_IAA(100uM)added2hrBefore',
    # --- 85kb (15C-A2) ---
    '85kb_None': '15C-A2_GSK',
    '85kb_IAA': '15C-A2_GSK_IAA(100uM)added2hrBefore',
    # --- 170kb (15B-18G9) ---
    '170kb_None': '15B-18G9_GSK',
    '170kb_IAA': '15B-18G9_GSK_IAA(100uM)added2hrBefore',
    # --- 255kb (15A-A6G9) ---
    '255kb_None': '15A-A6G9_GSK',
    '255kb_IAA': '15A-A6G9_GSK_IAA(100uM)added2hrBefore',
    # --- 340kb (S+V-A6B8) ---
    '340kb_None': 'S+V-A6B8_GSK',
    '340kb_IAA': 'S+V-A6B8_GSK_IAA(100uM)added2hrBefore',
    # --- 340kb_Cp (SCre-G5-H7) ---
    '340kb_Cp_None': 'SCre-G5-H7_GSK',
    # --- 340kb_Ce (VCre-C2b) ---
    '340kb_Ce_None': 'VCre-C2b_GSK',
    # --- 340kb_Ce_Cp (G7B8G2) ---
    '340kb_Ce_Cp_None': 'G7B8G2_GSK',
    '340kb_Ce_Cp_IAA': 'G7B8G2_GSK_100uM_IAAadded2hrBefore',
    '340kb_Ce_Cp_dTAG': 'G7B8G2_GSK_500nMdTAG13added2hrBefore',
    '340kb_Ce_Cp_IAAdTAG': 'G7B8G2_GSK_dTAG13(500uM)andIAA(100uM)added2hrBefore',
    # --- 340kb_Ce_Cp_noE_noP (E11G8) ---
    '340kb_Ce_Cp_noE_noP_None': 'E11G8_GSK',
    # --- 340kb_Ce_Cp_noE (Vika-H11) ---
    '340kb_Ce_Cp_noE_None': 'Vika-H11_GSK',
    # --- 2.362kb_noE_noP (14A-A11) ---
    '2.362kb_noE_noP_None': '14A-A11_GSK',
    # --- 0.395kb_noE_noP (14A-A11E6) ---
    '0.395kb_noE_noP_None': '14A-A11E6_GSK',
}

def filter_out_extreme_pos(data, mu=None, sig=None, significance_level=0.01):
    pval = significance_level / len(data[0])
    if mu is not None: mus_pos = mu
    else: mus_pos = np.nanmean(data, axis=(0, 1))
    if sig is not None: std_pos = sig
    else: std_pos = np.nanstd(data, axis=(0, 1))

    summand_pos = (data - mus_pos) ** 2 / std_pos**2
    chi_pos = np.nansum(summand_pos, axis=-1)
    pvals_pos = 1 - chi2.cdf(chi_pos, df=np.sum(~np.isnan(data), axis=-1))

    significant_pos = pvals_pos < pval
    filtered = np.where((significant_pos)[..., None], np.nan, data)
    return filtered, significant_pos

def filter_out_1frame_jumps(data, significance_level=0.01):
    resid = data[:, 1:-1] - 0.5 * (data[:, :-2] + data[:, 2:])
    ms = np.nanmean(resid, axis=(0, 1))
    ss = np.nanstd(resid, axis=(0, 1))
    z = np.abs(np.nansum((resid - ms) ** 2 / ss**2, axis=-1))

    is_local_max = (
        z >= np.pad(z[:, :-1], ((0, 0), (1, 0)), constant_values=-np.inf)
    ) & (z >= np.pad(z[:, 1:], ((0, 0), (0, 1)), constant_values=-np.inf))
    pval = 1 - chi2.cdf(z, df=3)
    nobs = np.sum(~np.isnan(data[..., 0]), axis=-1)[:, None]
    significant = pval < significance_level / nobs
    single_frame_jump = significant & is_local_max

    padded_single_frame_jump = np.zeros((single_frame_jump.shape[0], single_frame_jump.shape[1] + 2), dtype=bool)
    padded_single_frame_jump[:, 1:-1] = single_frame_jump
    filtered = np.where(padded_single_frame_jump[..., None], np.nan, data)
    return filtered, single_frame_jump

def filter_inconsistent_trajectories(ep_dat, mu, sig, significance_level=0.01):
    filtered, significant_pos = filter_out_extreme_pos(ep_dat, mu=mu, sig=sig, significance_level=significance_level)
    filtered, single_frame_jump = filter_out_1frame_jumps(filtered, significance_level=significance_level)
    rv = np.linalg.norm(ep_dat, axis=-1)
    rv_filtered = np.linalg.norm(filtered, axis=-1)
    padded_single_frame_jump = np.zeros((single_frame_jump.shape[0], single_frame_jump.shape[1] + 2), dtype=bool)
    padded_single_frame_jump[:, 1:-1] = single_frame_jump
    removed = (significant_pos | padded_single_frame_jump) & (~np.isnan(rv))
    return filtered, rv, rv_filtered, removed

def load_and_build_array(parent_dir, target_cond, condition_mapping):
    if not os.path.exists(parent_dir): return None

    movie_dirs = [d for d in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, d))]
    raw_frames = []

    for movie_name in movie_dirs:
        cond_match = re.search(r'^\d{8}_(.*?)(?:_30ms|30ms|_30_ms|_10ms|10ms|_10_ms)', movie_name)
        extracted_cond = cond_match.group(1).rstrip('_').strip() if cond_match else 'Unknown'

        mapped_cond = condition_mapping.get(extracted_cond, extracted_cond)

        if mapped_cond != target_cond:
            continue

        for csv_file in glob.glob(os.path.join(parent_dir, movie_name, '*_*.csv')):
            try:
                df = pd.read_csv(csv_file)
                df['TrackID'] = f"{movie_name}_{os.path.basename(csv_file).replace('.csv', '')}"
                df['rel_x'] = (df['pro_x (nm)'] - df['enh_x (nm)']) / 1000.0
                df['rel_y'] = (df['pro_y (nm)'] - df['enh_y (nm)']) / 1000.0
                df['rel_z'] = (df['pro_z (nm)'] - df['enh_z (nm)']) / 1000.0

                if 'intensity (au)' not in df.columns:
                    df['intensity (au)'] = np.nan

                raw_frames.append(df[['TrackID', 'frame', 'rel_x', 'rel_y', 'rel_z', 'intensity (au)']])
            except Exception: pass

    if not raw_frames: return None

    raw_df = pd.concat(raw_frames, ignore_index=True)
    min_f, max_f = raw_df['frame'].min(), raw_df['frame'].max()
    num_frames = int(max_f - min_f + 1)

    track_ids = raw_df['TrackID'].unique()
    track_idx_map = {tid: i for i, tid in enumerate(track_ids)}

    data_spat = np.full((len(track_ids), num_frames, 3), np.nan)
    data_int = np.full((len(track_ids), num_frames), np.nan)

    t_idxs = raw_df['TrackID'].map(track_idx_map).values
    f_idxs = (raw_df['frame'] - min_f).astype(int).values

    data_spat[t_idxs, f_idxs, 0] = raw_df['rel_x'].values
    data_spat[t_idxs, f_idxs, 1] = raw_df['rel_y'].values
    data_spat[t_idxs, f_idxs, 2] = raw_df['rel_z'].values
    data_int[t_idxs, f_idxs] = raw_df['intensity (au)'].values

    return data_spat, data_int

In [25]:
results = []

for npz_cond, user_cond in tqdm(NPZ_CONDITION_MAPPING.items(), desc="Validating Conditions"):
    for fr, raw_dir in [('30s', DIR_30S), ('5s', DIR_5S)]:
        npz_filename = f"{fr}_{npz_cond}.npz"
        npz_path = os.path.join(NPZ_DIR, npz_filename)

        if not os.path.exists(npz_path):
            continue

        npz_data = np.load(npz_path)['dataset']

        valid_tracks_mask_npz = ~np.isnan(npz_data[..., 0]).all(axis=1)
        n_traj_npz = valid_tracks_mask_npz.sum()
        n_frames_npz = np.sum(~np.isnan(npz_data[..., 0]))

        dist_npz = np.linalg.norm(npz_data, axis=-1) / 1000.0
        mean_dist_npz = np.nanmean(dist_npz)
        med_dist_npz = np.nanmedian(dist_npz)

        mean_int_npz = 'N/A'
        med_int_npz = 'N/A'

        raw_res = load_and_build_array(raw_dir, user_cond, CONDITION_MAPPING)

        if raw_res is None:
            n_traj_raw, n_frames_raw, mean_dist_raw, med_dist_raw = 0, 0, np.nan, np.nan
            mean_int_raw, med_int_raw = np.nan, np.nan
        else:
            raw_data_spat, raw_data_int = raw_res

            with np.errstate(all='ignore'):
                m, s = np.nanmean(raw_data_spat, axis=(0, 1)), np.nanstd(raw_data_spat, axis=(0, 1))
                if np.any(np.isnan(s)) or np.any(s == 0):
                    s = np.nanstd(raw_data_spat) * np.ones(3)
                    m = np.nanmean(raw_data_spat) * np.ones(3)

                filtered_raw_spat, _, _, _ = filter_inconsistent_trajectories(
                    raw_data_spat, mu=m, sig=s, significance_level=significance_level
                )

            valid_tracks_mask_raw = ~np.isnan(filtered_raw_spat[..., 0]).all(axis=1)
            n_traj_raw = valid_tracks_mask_raw.sum()
            n_frames_raw = np.sum(~np.isnan(filtered_raw_spat[..., 0]))

            dist_raw = np.linalg.norm(filtered_raw_spat, axis=-1)
            mean_dist_raw = np.nanmean(dist_raw)
            med_dist_raw = np.nanmedian(dist_raw)

            filtered_raw_int = np.where(np.isnan(filtered_raw_spat[..., 0]), np.nan, raw_data_int)
            mean_int_raw = np.nanmean(filtered_raw_int)
            med_int_raw = np.nanmedian(filtered_raw_int)

        results.append({
            'Condition': f"{fr} | {npz_cond}",
            'Traj (npz)': int(n_traj_npz),
            'Traj (Rebuild)': int(n_traj_raw),
            'Frames (npz)': int(n_frames_npz),
            'Frames (Rebuild)': int(n_frames_raw),
            'Mean Dist npz (um)': f"{mean_dist_npz:.4f}",
            'Mean Dist Rebuild (um)': f"{mean_dist_raw:.4f}",
            'Med Dist npz (um)': f"{med_dist_npz:.4f}",
            'Med Dist Rebuild (um)': f"{med_dist_raw:.4f}",
            'Intensity npz': mean_int_npz,
            'Mean Int Rebuild (au)': f"{mean_int_raw:.2f}" if not pd.isna(mean_int_raw) else "N/A",
            'Med Int Rebuild (au)': f"{med_int_raw:.2f}" if not pd.isna(med_int_raw) else "N/A"
        })

df_results = pd.DataFrame(results)

def color_match(val):
    return 'background-color: #E8F5E9' if val else 'background-color: #FFEBEE'

df_results['Traj Match'] = df_results['Traj (npz)'] == df_results['Traj (Rebuild)']
df_results['Frame Match'] = df_results['Frames (npz)'] == df_results['Frames (Rebuild)']

display(df_results)


Validating Conditions: 100%|██████████| 20/20 [00:08<00:00,  2.28it/s]


,Condition,Traj (npz),Traj (Rebuild),Frames (npz),Frames (Rebuild),Mean Dist npz (um),Mean Dist Rebuild (um),Med Dist npz (um),Med Dist Rebuild (um),Intensity npz,Mean Int Rebuild (au),Med Int Rebuild (au),Traj Match,Frame Match
0,30s | 1.5kb_None,100,100,24755,24755,0.1543,0.1543,0.1462,0.1462,N/A,9.69,5.76,True,True
1,30s | 1.5kb_IAA,9,9,2743,2743,0.1771,0.1771,0.1696,0.1696,N/A,2.03,0.08,True,True
2,30s | 85kb_None,111,111,40683,40683,0.2126,0.2126,0.1964,0.1964,N/A,3.76,1.51,True,True
3,5s | 85kb_None,125,125,50432,50432,0.2060,0.2060,0.1915,0.1915,N/A,3.94,1.33,True,True
4,30s | 85kb_IAA,36,36,7576,7576,0.3271,0.3271,0.3052,0.3052,N/A,0.45,0.31,True,True
5,30s | 170kb_None,146,146,72793,72793,0.2219,0.2219,0.2059,0.2059,N/A,3.39,1.53,True,True
6,5s | 170kb_None,153,153,71844,71844,0.2149,0.2149,0.1998,0.1998,N/A,3.54,1.68,True,True
7,30s | 170kb_IAA,23,23,6853,6853,0.3799,0.3799,0.3467,0.3467,N/A,1.03,0.49,True,True
8,30s | 255kb_None,182,182,79552,79552,0.2568,0.2568,0.2408,0.2408,N/A,2.19,0.91,True,True
9,5s | 255kb_None,153,153,61234,61234,0.2412,0.2412,0.2236,0.2236,N/A,2.11,0.77,True,True
